(1/6)
# Artemis II Trajectory Analysis
Compute and visualize the trajectory of the Artemis II mission using `astroquery`, `astropy`, and `matplotlib`.

In [ ]:
# Step 0: Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from datetime import timezone, timedelta

# Local timezone offset (UTC-5)
tz_local = timezone(timedelta(hours=-5))

# Hide warnings
import warnings
warnings.filterwarnings('ignore')

### Step 1: Connecting to NASA's JPL Horizons
Using the Orion capsule's specific ID (`-1024`) and calculate its position relative to the Earth's center (`399`). We are fetching data for the exact mission epoch we want to analyze.

In [ ]:
current_time = Time.now()
current_local = current_time.to_datetime(timezone=tz_local)

# Official Artemis II launch: April 1, 2026 at 22:35:12 UTC (6:35:12 PM EDT)
mission_start = Time("2026-04-01 22:35:12")

# JPL Horizons ephemeris availability window for Orion (-1024)
traj_start = Time("2026-04-02 01:59:00")  # earliest available: ~01:58:32 TD
traj_end   = Time("2026-04-10 23:50:00")  # latest available:  ~23:54:30 TD

# Orion (-1024) and Moon (301) relative to Earth (399)
print(f"Current UTC:   {current_time.iso}")
print(f"Current Local: {current_local.strftime('%Y-%m-%d %H:%M:%S')} (UTC-5)")
print("Connecting to JPL Horizons database...")
orion_obj = Horizons(id='-1024', location='399', epochs=current_time.jd)
moon_obj = horizons_moon = Horizons(id='301', location='399', epochs=current_time.jd)

print("Connection established!")

### Step 2: Getting Current Telemetry (Raw)
Fetch the ephemeris state vectors. By default, Horizons returns these values in Astronomical Units (AU) and Astronomical Units per Day (AU/day).

In [ ]:
# Fetch raw vector tables
orion_vecs = orion_obj.vectors()
moon_vecs = moon_obj.vectors()

# Extract the position (x, y, z) and velocity (vx, vy, vz) components
r_orion_raw = np.array([orion_vecs['x'][0], orion_vecs['y'][0], orion_vecs['z'][0]])
v_orion_raw = np.array([orion_vecs['vx'][0], orion_vecs['vy'][0], orion_vecs['vz'][0]])

print(f"Orion Raw Position (AU): {r_orion_raw}")
print(f"Orion Raw Velocity (AU/d): {v_orion_raw}")

### Step 3: Convert State Vectors to SI Units
Attach Astropy units to the raw data and cleanly convert them to Kilometers (km) and Kilometers per second (km/s).

In [ ]:
r_orion = (r_orion_raw * u.au).to(u.km)
v_orion = (v_orion_raw * u.au / u.d).to(u.km / u.s)

r_moon = (np.array([moon_vecs['x'][0], moon_vecs['y'][0], moon_vecs['z'][0]]) * u.au).to(u.km)

print(f"Orion Position (SI): {np.round(r_orion.value, 2)} km")
print(f"Orion Velocity (SI): {np.round(v_orion.value, 2)} km/s")

### Step 4: Include All Telemetry Values
Calculate the real world metrics : Mission Elapsed Time, Distance to Earth, Distance to Moon, and Speed.

In [ ]:
# 1. Mission Elapsed Time (MET)
elapsed = current_time - mission_start
days = int(elapsed.jd)
hours = int((elapsed.jd - days) * 24)
minutes = int((((elapsed.jd - days) * 24) - hours) * 60)
flight_day = days + 1
elapsed_time_str = f"T+ {days}d {hours}h {minutes}m"

# 2. Distance Calculations 
# Magnitude of the Orion vector (Distance to Earth Center)
dist_earth_km = np.linalg.norm(r_orion).value # norm between vectors
dist_earth_mi = dist_earth_km * 0.621371

# Magnitude of Orion vector minus Moon vector (Distance to Moon)
dist_moon_km = np.linalg.norm(r_orion - r_moon).value # norm between vectors
dist_moon_mi = dist_moon_km * 0.621371

# 3. Speed Calculation
speed_kmh = np.linalg.norm(v_orion).to(u.km / u.h).value
speed_mph = speed_kmh * 0.621371

print("==== ARTEMIS II TELEMETRY ====")
print(f"Timestamp UTC:       {current_time.iso}")
print(f"Timestamp Local:     {current_local.strftime('%Y-%m-%d %H:%M:%S')} (UTC-5)")
print(f"Flight Day:          Day {flight_day}")
print(f"Mission Elapsed Time: {elapsed_time_str}")
print(f"Distance to Earth:   {dist_earth_km:,.2f} km  ({dist_earth_mi:,.2f} mi)")
print(f"Distance to Moon:    {dist_moon_km:,.2f} km  ({dist_moon_mi:,.2f} mi)")
print(f"Current Speed:       {speed_kmh:,.2f} km/h  ({speed_mph:,.2f} mph)")
print("==============================")

### Step 5: Getting the Full Trajectory
Query an array of positions for the entire 10-day flight plan.

In [ ]:
print("Fetching full orbital trajectory...")

# Coarse timeframe for plotting (1-hour steps — fewer points, cleaner lines)
timeframe_plot = {
    'start': traj_start.iso,
    'stop': traj_end.iso,
    'step': '1h'
}

# Fine timeframe for closest-approach calculation (1-min steps)
timeframe_fine = {
    'start': traj_start.iso,
    'stop': traj_end.iso,
    'step': '1min'
}

# All queries from Solar System Barycenter (@0), then subtract Earth for geocentric
# This produces smoother trajectories than direct geocentric queries

# --- Coarse data for plotting ---
orion_ssb_plot = Horizons(id='-1024', location='@0', epochs=timeframe_plot).vectors()
moon_ssb_plot  = Horizons(id='301',   location='@0', epochs=timeframe_plot).vectors()
earth_ssb_plot = Horizons(id='399',   location='@0', epochs=timeframe_plot).vectors()

# Geocentric = SSB - Earth_SSB
orion_x = ((np.array(orion_ssb_plot['x']) - np.array(earth_ssb_plot['x'])) * u.au).to(u.km).value
orion_y = ((np.array(orion_ssb_plot['y']) - np.array(earth_ssb_plot['y'])) * u.au).to(u.km).value

moon_x = ((np.array(moon_ssb_plot['x']) - np.array(earth_ssb_plot['x'])) * u.au).to(u.km).value
moon_y = ((np.array(moon_ssb_plot['y']) - np.array(earth_ssb_plot['y'])) * u.au).to(u.km).value

# Keep full vectors for 3D later
orion_full = orion_ssb_plot
moon_full  = moon_ssb_plot
earth_full = earth_ssb_plot

print(f"Plot data fetched: {len(orion_x)} points")

# Full Moon orbit via SSB (~28 days for the sweeping arc context)
print("Fetching full Moon orbit from SSB...")
moon_orbit_start = Time("2026-03-22 00:00:00")
moon_orbit_end = Time("2026-04-19 00:00:00")
moon_orbit_epochs = {
    'start': moon_orbit_start.iso,
    'stop': moon_orbit_end.iso,
    'step': '1h'
}

moon_ssb_orbit  = Horizons(id='301', location='@0', epochs=moon_orbit_epochs).vectors()
earth_ssb_orbit = Horizons(id='399', location='@0', epochs=moon_orbit_epochs).vectors()

moon_orbit_x = ((np.array(moon_ssb_orbit['x']) - np.array(earth_ssb_orbit['x'])) * u.au).to(u.km).value
moon_orbit_y = ((np.array(moon_ssb_orbit['y']) - np.array(earth_ssb_orbit['y'])) * u.au).to(u.km).value
moon_orbit_z = ((np.array(moon_ssb_orbit['z']) - np.array(earth_ssb_orbit['z'])) * u.au).to(u.km).value
print(f"Moon orbit fetched: {len(moon_orbit_x)} points")

# --- Fine data for closest approach (also from SSB) ---
print("Fetching fine-resolution data for closest approach...")
orion_ssb_fine = Horizons(id='-1024', location='@0', epochs=timeframe_fine).vectors()
moon_ssb_fine  = Horizons(id='301',   location='@0', epochs=timeframe_fine).vectors()
earth_ssb_fine = Horizons(id='399',   location='@0', epochs=timeframe_fine).vectors()

orion_fine = orion_ssb_fine
moon_fine  = moon_ssb_fine
earth_fine = earth_ssb_fine
print(f"Fine data fetched: {len(orion_fine)} points")

### Step 5b: Closest Approach to the Moon
Compute when Orion reaches its minimum distance to the Moon during the mission, and how long until (or since) that moment.

In [ ]:
# Compute geocentric positions from SSB data (subtract Earth)
orion_xyz = np.column_stack([
    ((np.array(orion_fine['x']) - np.array(earth_fine['x'])) * u.au).to(u.km).value,
    ((np.array(orion_fine['y']) - np.array(earth_fine['y'])) * u.au).to(u.km).value,
    ((np.array(orion_fine['z']) - np.array(earth_fine['z'])) * u.au).to(u.km).value,
])
moon_xyz = np.column_stack([
    ((np.array(moon_fine['x']) - np.array(earth_fine['x'])) * u.au).to(u.km).value,
    ((np.array(moon_fine['y']) - np.array(earth_fine['y'])) * u.au).to(u.km).value,
    ((np.array(moon_fine['z']) - np.array(earth_fine['z'])) * u.au).to(u.km).value,
])

distances_to_moon = np.linalg.norm(orion_xyz - moon_xyz, axis=1)

# Find closest approach
min_idx = np.argmin(distances_to_moon)
min_dist_km = distances_to_moon[min_idx]
min_dist_mi = min_dist_km * 0.621371

# Get the time of closest approach from the ephemeris JD column
closest_jd = orion_fine['datetime_jd'][min_idx]
closest_time = Time(closest_jd, format='jd')
closest_local = closest_time.to_datetime(timezone=tz_local)

# Time difference from now
delta = closest_time - current_time
delta_hours = delta.to(u.h).value

print("==== CLOSEST LUNAR APPROACH ====")
print(f"Time of Closest Approach (UTC):   {closest_time.iso}")
print(f"Time of Closest Approach (Local): {closest_local.strftime('%Y-%m-%d %H:%M:%S')} (UTC-5)")
print(f"Minimum Distance:                 {min_dist_km:,.2f} km  ({min_dist_mi:,.2f} mi)")

if delta_hours > 0:
    d = int(delta_hours // 24)
    h = int(delta_hours % 24)
    m = int((delta_hours % 1) * 60)
    print(f"Time Until Closest Approach: {d}d {h}h {m}m from now")
else:
    past_hours = abs(delta_hours)
    d = int(past_hours // 24)
    h = int(past_hours % 24)
    m = int((past_hours % 1) * 60)
    print(f"Closest Approach Was: {d}d {h}h {m}m ago")
print("================================")

### Step 6: Plotting the Scene
Finally, we use `matplotlib` to render strings of data points into a beautiful 2D mission control map. 
Split the trajectory into "Past" (solid lines) and "Future" (dotted lines) based on the `current_time`.

In [ ]:
# Interpolate with cubic splines for smooth curves through real data points
from scipy.interpolate import make_interp_spline

t = np.arange(len(orion_x))
t_smooth = np.linspace(0, len(orion_x) - 1, len(orion_x) * 10)

orion_x_s = make_interp_spline(t, orion_x, k=3)(t_smooth)
orion_y_s = make_interp_spline(t, orion_y, k=3)(t_smooth)

# Calculate split index for Past/Future paths (scaled to interpolated array)
elapsed_from_traj_start = (current_time - traj_start).jd
split_idx_raw = max(0, min(int(elapsed_from_traj_start * 24), len(orion_x) - 1))
split_idx = split_idx_raw * 10  # scale to interpolated array

# Moon orbit split index (hourly data from moon_orbit_start)
moon_orbit_split = max(0, min(int((current_time - moon_orbit_start).jd * 24), len(moon_orbit_x) - 1))

# Current Moon position from the mission-period data
moon_now_x = moon_x[split_idx_raw]
moon_now_y = moon_y[split_idx_raw]

fig, ax = plt.subplots(figsize=(10, 10), facecolor='#050505')
ax.set_facecolor('#050505')

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(colors='#333', bottom=False, left=False)

# 1. Plot Earth (Center)
ax.plot(0, 0, 'o', color='#2a75d3', markersize=14)
ax.annotate("EARTH", (0, 0), textcoords="offset points", xytext=(10,10), color='#2a75d3', fontweight='heavy')

# 2. Plot Moon's orbit arc — past (solid) + future (dotted)
ax.plot(moon_orbit_x[:moon_orbit_split+1], moon_orbit_y[:moon_orbit_split+1], color='gray', linestyle='-', alpha=0.2, linewidth=1)
ax.plot(moon_orbit_x[moon_orbit_split:], moon_orbit_y[moon_orbit_split:], color='gray', linestyle=':', alpha=0.15, linewidth=1)

# 3. Moon current position
ax.plot(moon_now_x, moon_now_y, 'o', color='gray', markersize=12)
ax.annotate("MOON", (moon_now_x, moon_now_y), textcoords="offset points", xytext=(8,-12), color='gray', fontweight='bold')

# 4. Plot Orion's Trajectory & Current Position
# Past: Solid cyan line with a faint glow
ax.plot(orion_x_s[:split_idx+1], orion_y_s[:split_idx+1], color='#00d4ff', linewidth=2, alpha=0.8)
ax.plot(orion_x_s[:split_idx+1], orion_y_s[:split_idx+1], color='#00d4ff', linewidth=4, alpha=0.2)
# Future: Dotted cyan line
ax.plot(orion_x_s[split_idx:], orion_y_s[split_idx:], color='#00d4ff', linestyle=':', alpha=0.4)
# Current Pos: Red dot (use raw position for accuracy)
ax.plot(orion_x[split_idx_raw], orion_y[split_idx_raw], 'o', color='#ff3333', markersize=10)
ax.annotate("ORION", (orion_x[split_idx_raw], orion_y[split_idx_raw]), textcoords="offset points", xytext=(8,-12), color='#ff3333', fontweight='heavy')

plt.title(f"Artemis II Trajectory | {elapsed_time_str}", color='white', fontsize=16)
ax.set_aspect('equal', 'datalim')
plt.tight_layout()
plt.show()

### Step 7: Interactive 3D Visualization (Plotly)
Use Plotly's `Scatter3d` to render a rotatable 3D view of the Orion and Moon trajectories around Earth.

In [ ]:
import plotly.graph_objects as go

# Build full 3D geocentric arrays (SSB - Earth)
orion_z = ((np.array(orion_full['z']) - np.array(earth_full['z'])) * u.au).to(u.km).value
moon_z  = ((np.array(moon_full['z'])  - np.array(earth_full['z'])) * u.au).to(u.km).value

orion_z_s = make_interp_spline(t, orion_z, k=3)(t_smooth)

# Current Moon z from mission period
moon_now_z = moon_z[split_idx_raw]

fig = go.Figure()

# Earth glow layers (outer to inner)
for glow_size, glow_opacity in [(28, 0.06), (22, 0.10), (16, 0.18)]:
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode='markers',
        marker=dict(size=glow_size, color='#4a9eff', opacity=glow_opacity),
        showlegend=False, hoverinfo='skip'
    ))

# Earth
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers+text',
    marker=dict(size=10, color='#2a75d3', line=dict(width=2, color='#6ab0ff')),
    text=['Earth'], textposition='top center',
    textfont=dict(color='#6ab0ff', size=13, family='Arial Black'), name='Earth',
    showlegend=False
))

# Moon orbit — past (solid) + future (dotted)
fig.add_trace(go.Scatter3d(
    x=moon_orbit_x[:moon_orbit_split+1], y=moon_orbit_y[:moon_orbit_split+1], z=moon_orbit_z[:moon_orbit_split+1],
    mode='lines', line=dict(color='#666666', width=2), name='Moon orbit (past)', opacity=0.3,
    showlegend=False
))
fig.add_trace(go.Scatter3d(
    x=moon_orbit_x[moon_orbit_split:], y=moon_orbit_y[moon_orbit_split:], z=moon_orbit_z[moon_orbit_split:],
    mode='lines', line=dict(color='#666666', width=2, dash='dot'), name='Moon orbit (future)', opacity=0.2,
    showlegend=False
))

# Moon glow
fig.add_trace(go.Scatter3d(
    x=[moon_now_x], y=[moon_now_y], z=[moon_now_z],
    mode='markers', marker=dict(size=16, color='#aaaaaa', opacity=0.12),
    showlegend=False, hoverinfo='skip'
))

# Moon current position
fig.add_trace(go.Scatter3d(
    x=[moon_now_x], y=[moon_now_y], z=[moon_now_z],
    mode='markers+text', marker=dict(size=8, color='#cccccc', line=dict(width=1, color='white')),
    text=['Moon'], textposition='top center',
    textfont=dict(color='#cccccc', size=12, family='Arial Black'), name='Moon',
    showlegend=False
))

# Orion trajectory — past (bright glow + core)
fig.add_trace(go.Scatter3d(
    x=orion_x_s[:split_idx+1], y=orion_y_s[:split_idx+1], z=orion_z_s[:split_idx+1],
    mode='lines', line=dict(color='#00d4ff', width=8), name='Orion (past) glow', opacity=0.15,
    showlegend=False
))
fig.add_trace(go.Scatter3d(
    x=orion_x_s[:split_idx+1], y=orion_y_s[:split_idx+1], z=orion_z_s[:split_idx+1],
    mode='lines', line=dict(color='#00d4ff', width=5), name='Orion (past)', opacity=0.9,
    showlegend=False
))

# Orion trajectory — future (dashed)
fig.add_trace(go.Scatter3d(
    x=orion_x_s[split_idx:], y=orion_y_s[split_idx:], z=orion_z_s[split_idx:],
    mode='lines', line=dict(color='#00d4ff', width=3, dash='dot'), name='Orion (future)', opacity=0.5,
    showlegend=False
))

# Orion current position — glow + marker
fig.add_trace(go.Scatter3d(
    x=[orion_x[split_idx_raw]], y=[orion_y[split_idx_raw]], z=[orion_z[split_idx_raw]],
    mode='markers', marker=dict(size=14, color='#ff3333', opacity=0.15),
    showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter3d(
    x=[orion_x[split_idx_raw]], y=[orion_y[split_idx_raw]], z=[orion_z[split_idx_raw]],
    mode='markers+text', marker=dict(size=7, color='#ff4444', line=dict(width=2, color='#ff8888')),
    text=['Orion'], textposition='top center',
    textfont=dict(color='#ff6666', size=12, family='Arial Black'), name='Orion (now)',
    showlegend=False
))

# Time markers along Orion's trajectory (every 24 hours)
pts_per_day = 24 * 10  # 10x upsampled from 1h data
for day in range(1, int(len(orion_x_s) / pts_per_day) + 1):
    idx = day * pts_per_day
    if idx < len(orion_x_s):
        fig.add_trace(go.Scatter3d(
            x=[orion_x_s[idx]], y=[orion_y_s[idx]], z=[orion_z_s[idx]],
            mode='markers+text', marker=dict(size=3, color='#00d4ff', symbol='diamond'),
            text=[f'Day {day}'], textposition='bottom center',
            textfont=dict(color='#00a0cc', size=9), showlegend=False,
            hovertext=f'Mission Day {day + 1}'
        ))

fig.update_layout(
    title=dict(
        text=f'Artemis II — 3D Trajectory | {elapsed_time_str}',
        font=dict(color='#00d4ff', size=18, family='Arial Black'),
        x=0.5
    ),
    scene=dict(
        xaxis=dict(title='X (km)', color='#555', gridcolor='#1a1a1a', showbackground=True,
                    backgroundcolor='#080808', gridwidth=1, zerolinecolor='#222'),
        yaxis=dict(title='Y (km)', color='#555', gridcolor='#1a1a1a', showbackground=True,
                    backgroundcolor='#080808', gridwidth=1, zerolinecolor='#222'),
        zaxis=dict(title='Z (km)', color='#555', gridcolor='#1a1a1a', showbackground=True,
                    backgroundcolor='#080808', gridwidth=1, zerolinecolor='#222'),
        bgcolor='#050505',
        aspectmode='data',
    ),
    paper_bgcolor='#050505',
    showlegend=False,
    width=1000, height=800,
    margin=dict(l=0, r=0, t=50, b=30),
)
fig.show()